# Stage 1 — Frozen backbone embeddings (no training)

### Goal & Description

**Goal:**
1) Clean data: To sanitize the raw dataset before training by using a frozen foundation model. It should identify duplicate image bursts, flag low-quality outliers, and mathematically determine the Train/Test splits.
2) Backbone Choice: Find the best "baseline" space (with frozen models)

**Input:** 
    * cropped images
    * foreground RGBA
    * blurred background

**Model:** frozen
    * DINOv2 (ViT)
    * MegaDescriptor-L-384 (Swin-B Transformer)


**Output:** embeddings per image - saved under data export folder

**Evaluation:** Rank-1
(also try to concatenate embeddings)


-> diagnostic stage: No learning yet.


### Open Question:
1) What inputs should be fed to backbone model - chatty and google AI disagree if blurred background, crop or foreground rgba
2) Can we concatenate embeddings from different models? And would it make it better? e.g. np.hstack([dinov2, mega]))


### Resources:
**Mega Descriptor**
* https://huggingface.co/BVRA/MegaDescriptor-L-384
* https://hackernoon.com/wildlifedatasets-an-open-source-toolkit-for-animal-re-identification-megadescriptor-methodology
* https://huggingface.co/BVRA/MegaDescriptor-DINOv2-518

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "51"

In [ ]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/Colab Notebooks/Applied Computer Vision/Project_Scratchpad Ideas/"

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
# Install dependencies
%%capture
%pip install --no-cache-dir -r requirements.txt

In [ ]:
import numpy as np
from PIL import Image
import json

import torch
from torch.utils.data import Dataset, DataLoader

import fiftyone as fo
import fiftyone.brain as fob

from tqdm import tqdm

In [ ]:
from config import NUM_WORKERS, DEVICE, SEED
from utils import ensure_dir
from FiftyOne import FODataset
from models import load_dinov2, load_mega_descriptor, load_dinov2_for_wildlife
from datasets import JaguarDataset

In [ ]:
DATASET_NAME = "jaguar_stage1_embeddings"  
FODATASET_DIR_NAME = "jaguar_export"       

MODEL_NAME = "dinov2_vits14"             # good default (384-d). options: dinov2_vitb14 (768-d), dinov2_vitl14, ...
BATCH_SIZE = 16

In [ ]:
# set paths
DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Applied Computer Vision/Project_Scratchpad Ideas")
WORK_ROOT = Path("/content")

DATA_ROOT = DRIVE_ROOT / f"datasets/{FODATASET_DIR_NAME}"
OUT_DIR = WORK_ROOT / "out"

ensure_dir([OUT_DIR, DATA_ROOT])

# Load Models

In [ ]:
def load_models():
    dinov2, dinov2_transform = load_dinov2(dinoname="dinov2_vitb14", frozen=True)
    mega_descriptor, mega_transform = load_mega_descriptor(frozen=True)
    #dinov2_wildlife, dinov2_wildlife_transform = load_dinov2_for_wildlife(frozen=True)

    return [
        ( "DINOv2_Base", dinov2, dinov2_transform, ),
        ( "MegaDescriptor_L", mega_descriptor, mega_transform, ),
        #( "DINOv2_Wildlife", dinov2_wildlife, dinov2_wildlife_transform, )
    ]

# Create and save Embeddings

In [ ]:
# create dataset
with open(DATA_ROOT / "samples.json", "r") as f:
    samples = json.load(f)["samples"]

dataset = JaguarDataset(
    base_root=DATA_ROOT,
    transform=None,
    #processing_fn=my_blur # or None for raw crops
)
# Inject the pre-loaded samples so it doesn't re-read the file
dataset.samples = samples 

In [ ]:
@torch.no_grad
def create_embeddings(model, loader, device=DEVICE):
    all_ids = []
    all_filepaths = []
    all_embs = []

    for batch in tqdm(loader):
        x = batch["img"].to(device, non_blocking=True)

        out = model(x)

        # Some variants return a dict; handle both safely:
        if isinstance(out, dict):
            # common keys: "x_norm_clstoken" or similar
            if "x_norm_clstoken" in out:
                emb = out["x_norm_clstoken"]
            elif "cls_token" in out:
                emb = out["cls_token"]
            else:
                # fallback: take first token if it's token sequence
                emb = out[list(out.keys())[0]]
                if emb.ndim == 3:
                    emb = emb[:, 0]
        else:
            emb = out
            if emb.ndim == 3:  # token sequence -> take CLS
                emb = emb[:, 0]

        emb = emb.float().cpu().numpy()

        all_ids.extend(batch["id"])
        all_filepaths.extend(batch["filepath"])
        all_embs.append(emb)

    embs = np.concatenate(all_embs, axis=0)
    print("Embeddings shape:", embs.shape)  # (N, D)

    return all_ids, all_filepaths, embs

In [ ]:
def save_embeddings(name, manipulation_mode, embs, ids, filepaths, save_dir="/content"):
    
    file_name = f"embeddings_{name}_{manipulation_mode}.npy"
    
    np.save(save_dir / file_name, embs)

    if not (save_dir / "sample_ids.npy").exists():
        np.save(save_dir / "sample_ids.npy", np.array(ids, dtype=object))
        np.save(save_dir / "sample_filepaths.npy", np.array(filepaths, dtype=object))

    print("Saved to:", save_dir.resolve())

    return

In [ ]:
manipulation_mode = "original"

save_dir = DATA_ROOT / "embeddings"
ensure_dir([save_dir])

emb_files = list(save_dir.glob("embeddings_*.npy"))

if not emb_files: 
    models = load_models()
    for name, model, transform in models:

        # swaps the transform on the existing dataset object
        dataset.transform = transform

        # Creates a new loader
        current_loader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True
        )

        ids, filepaths, emb = create_embeddings(model, current_loader)
        save_embeddings(name, manipulation_mode, emb, ids, filepaths, save_dir=save_dir)

        print(f"Embeddings for {name} created")

# Create a hybrid embedding between two models:

1) L2-Normalize Model A.
2) L2-Normalize Model B.
3) Concatenate them.
4) L2-Normalize the result (optional but recommended for Cosine Similarity).

In [ ]:
model_name = "DINOv2_Base"
manipulation_mode = "original"

In [ ]:
# Load specific embeddings
def load_embeddings(model_name, manipulation_mode, save_dir):
    """Loads specific embeddings and their corresponding identifiers."""
    save_dir = Path(save_dir)
    
    emb_file = save_dir / f"embeddings_{model_name}_{manipulation_mode}.npy"
    if not emb_file.exists():
        print(f"❌ Error: Could not find {emb_file}")
        return None
    
    # These usually stay the same across runs if the dataset didn't change          !!TODO: check than what happens with different manipulation modes
    ids_file = save_dir / "sample_ids.npy"
    fps_file = save_dir / "sample_filepaths.npy" 

    # Load the files
    # allow_pickle=True is necessary because sample_ids and filepaths are stored as Python 'object' arrays (strings/IDs)
    embs = np.load(emb_file)
    ids = np.load(ids_file, allow_pickle=True)
    filepaths = np.load(fps_file, allow_pickle=True)
    
    print(f"✅ Loaded {model_name} ({manipulation_mode}): {embs.shape}")

    return embs, ids, filepaths

In [ ]:
def create_hybrid_embeddings(embs_a, embs_b):
    """
    Combines two different embedding sets into one.
    Assumes embs_a and embs_b are numpy arrays of shape (N, D1) and (N, D2)
    """
    # 1. Normalize each individually
    # We add a tiny epsilon (1e-12) to avoid division by zero
    norm_a = embs_a / (np.linalg.norm(embs_a, axis=1, keepdims=True) + 1e-12)
    norm_b = embs_b / (np.linalg.norm(embs_b, axis=1, keepdims=True) + 1e-12)
    
    # 2. Concatenate (Stack horizontally)
    # If A is 768-d and B is 384-d, hybrid is 1152-d
    hybrid = np.hstack([norm_a, norm_b])
    
    # 3. Final normalization so the new vector has a magnitude of 1
    final_hybrid = hybrid / (np.linalg.norm(hybrid, axis=1, keepdims=True) + 1e-12)
    
    return final_hybrid

In [ ]:
manipulation_mode = "original"

# create a specific hybrid embedding
dino_original_embs, ids, filepaths = load_embeddings("DINOv2_Base", manipulation_mode, save_dir)
mega_original_embs, _, _ = load_embeddings("MegaDescriptor_L", manipulation_mode, save_dir)

hybrid_embs = create_hybrid_embeddings(dino_original_embs, mega_original_embs)
save_embeddings("Hybrid_Dino_MegaDescriptor", manipulation_mode, hybrid_embs, ids, filepaths, save_dir=save_dir)

# Visualize embeddings in FiftyOne

In [ ]:
# Import and load the FiftyOne Dataset from export folder
fo_dataset = FODataset.load_from_export(
    export_dir= DATA_ROOT,
    dataset_name=DATASET_NAME
)
ds = fo_dataset.get_dataset()

print(ds, ds.count())
print(ds.get_field_schema().keys())

In [ ]:
manipulation_mode = "original"
model_names = ["DINOv2_Base", "MegaDescriptor_L", "Hybrid_Dino_MegaDescriptor"]

for name in model_names:
    embs, _, fps = load_embeddings(name, manipulation_mode, save_dir)

    # Attach embedding to FiftyOne
    field_name = f"emb_{name.lower()}"

    # Map: filepath -> fiftyone_id
    filepath_to_id = { f: i for f, i in zip(ds.values("filepath"), ds.values("id"))}

    # order FiftyOne ids according to filepath list
    ordered_ids = [filepath_to_id[fp] for fp in fps if fp in filepath_to_id]
    
    # Compute Visualizations (UMAP)
    fob.compute_visualization(
        ds,
        embeddings=field_name,
        brain_key=f"viz_{name.lower()}",
        method="umap",
        seed=SEED
    )

    # Compute Similarity Index
    fob.compute_similarity(
        ds,
        embeddings=field_name,
        brain_key=f"sim_{name.lower()}"
    )

print("✅ All embeddings synced and visualizations computed.")

In [ ]:
session = fo_dataset.launch()
print(session.url)